In [1]:
import pandas as pd
import matplotlib.pyplot as mp
import sqlite3
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [2]:
df = pd.read_csv("netflix_titles.csv",encoding = 'latin1')

In [3]:
df.shape

(8807, 12)

In [29]:
df.tail(30)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,added_year,added_month,added_month_name,content_age,duration_minutes,seasons
8797,s8798,TV Show,Zak Storm,unknown,"Michael Johnston, Jessica Gee-George, Christin...",France,2018-09-13,2016,TV-Y7,3 Seasons,Kids' TV,Teen surfer Zak Storm is mysteriously transpor...,2018.0,9.0,September,2.0,NaN,3.0
8797,s8798,TV Show,Zak Storm,unknown,"Michael Johnston, Jessica Gee-George, Christin...",South Korea,2018-09-13,2016,TV-Y7,3 Seasons,Kids' TV,Teen surfer Zak Storm is mysteriously transpor...,2018.0,9.0,September,2.0,NaN,3.0
8797,s8798,TV Show,Zak Storm,unknown,"Michael Johnston, Jessica Gee-George, Christin...",Indonesia,2018-09-13,2016,TV-Y7,3 Seasons,Kids' TV,Teen surfer Zak Storm is mysteriously transpor...,2018.0,9.0,September,2.0,NaN,3.0
8798,s8799,Movie,Zed Plus,Chandra Prakash Dwivedi,"Adil Hussain, Mona Singh, K.K. Raina, Sanjay M...",India,2019-12-31,2014,TV-MA,131 min,Comedies,A philandering small-town mechanic's political...,2019.0,12.0,December,5.0,131.0,NaN
8798,s8799,Movie,Zed Plus,Chandra Prakash Dwivedi,"Adil Hussain, Mona Singh, K.K. Raina, Sanjay M...",India,2019-12-31,2014,TV-MA,131 min,Dramas,A philandering small-town mechanic's political...,2019.0,12.0,December,5.0,131.0,NaN
8798,s8799,Movie,Zed Plus,Chandra Prakash Dwivedi,"Adil Hussain, Mona Singh, K.K. Raina, Sanjay M...",India,2019-12-31,2014,TV-MA,131 min,International Movies,A philandering small-town mechanic's political...,2019.0,12.0,December,5.0,131.0,NaN
8799,s8800,Movie,Zenda,Avadhoot Gupte,"Santosh Juvekar, Siddharth Chandekar, Sachit P...",India,2018-02-15,2009,TV-14,120 min,Dramas,A change in the leadership of a political part...,2018.0,2.0,February,9.0,120.0,NaN
8799,s8800,Movie,Zenda,Avadhoot Gupte,"Santosh Juvekar, Siddharth Chandekar, Sachit P...",India,2018-02-15,2009,TV-14,120 min,International Movies,A change in the leadership of a political part...,2018.0,2.0,February,9.0,120.0,NaN
8800,s8801,TV Show,Zindagi Gulzar Hai,unknown,"Sanam Saeed, Fawad Khan, Ayesha Omer, Mehreen ...",Pakistan,2016-12-15,2012,TV-PG,1 Season,International TV Shows,"Strong-willed, middle-class Kashaf and carefre...",2016.0,12.0,December,4.0,NaN,1.0
8800,s8801,TV Show,Zindagi Gulzar Hai,unknown,"Sanam Saeed, Fawad Khan, Ayesha Omer, Mehreen ...",Pakistan,2016-12-15,2012,TV-PG,1 Season,Romantic TV Shows,"Strong-willed, middle-class Kashaf and carefre...",2016.0,12.0,December,4.0,NaN,1.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [6]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description'], dtype='object')

In [7]:
df['director'] = df['director'].fillna('unknown')

In [8]:
df['cast'] = df['cast'].fillna('unknown')
df['country'] = df['country'].fillna('unknown')
df['rating'] = df['rating'].fillna('unknown')
df['duration'] = df['duration'].fillna('unknown')

In [9]:
df['date_added'] = pd.to_datetime(df['date_added'],errors = 'coerce')

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8528\44827034.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date_added'] = pd.to_datetime(df['date_added'],errors = 'coerce')


In [10]:
df['country'] = df['country'].str.split(',')

In [11]:
#Remove extra spaces
df['country'] = df['country'].apply(lambda x: [c.strip() for c in x])
#Explode into multiple rows
df = df.explode('country')

In [12]:
df['listed_in'] = df['listed_in'].str.split(',')
df['listed_in'] = df['listed_in'].apply(lambda x: [c.strip() for c in x])
df = df.explode('listed_in')

In [13]:
df['added_year']  = df['date_added'].dt.year

In [14]:
df['added_month']  = df['date_added'].dt.month

In [15]:
df['added_month_name'] = df['date_added'].dt.month_name()

In [16]:
df['content_age'] = df['added_year'] - df['release_year']


In [17]:
# extract numeric part
df['duration_value'] = df['duration'].str.extract(r'(\d+)').astype(float)

# create movie duration column
df['duration_minutes'] = df.apply(
    lambda x: x['duration_value'] if x['type'] == 'Movie' else None,
    axis=1
)

# create tv show seasons column
df['seasons'] = df.apply(
    lambda x: x['duration_value'] if x['type'] == 'TV Show' else None,
    axis=1
)


In [18]:
df = df.drop('duration_value', axis=1)

In [19]:
conn = sqlite3.connect('dfanalysis.db')

In [20]:
df.to_sql('netflix_cleaned',conn,if_exists='replace',index=False)

23764

In [21]:
conn.execute("DROP TABLE IF EXISTS df;")
conn.commit()


In [22]:
type_ = conn.execute("""SELECT 
    type,
    COUNT(*) AS total_count
FROM netflix_cleaned
GROUP BY type;""")

In [68]:
df.to_csv("netflix_cleaned.csv", index=False)

In [24]:
all = conn.execute('select * from netflix_cleaned')

In [25]:
all.fetchall()

[('s1',
  'Movie',
  'Dick Johnson Is Dead',
  'Kirsten Johnson',
  'unknown',
  'United States',
  '2021-09-25 00:00:00',
  2020,
  'PG-13',
  '90 min',
  'Documentaries',
  'As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.',
  2021.0,
  9.0,
  'September',
  1.0,
  90.0,
  None),
 ('s2',
  'TV Show',
  'Blood & Water',
  'unknown',
  'Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng',
  'South Africa',
  '2021-09-24 00:00:00',
  2021,
  'TV-MA',
  '2 Seasons',
  'International TV Shows',
  'After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at bi

In [26]:
conn.close()


In [33]:
df.duplicated().sum()


np.int64(0)

In [32]:
df['duration_minutes'].describe()
df['seasons'].describe()


count    6881.000000
mean        1.757739
std         1.594556
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        17.000000
Name: seasons, dtype: float64

In [34]:
df[df['content_age'] < 0]


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,added_year,added_month,added_month_name,content_age,duration_minutes,seasons
1551,s1552,TV Show,Hilda,unknown,"Bella Ramsey, Ameerah Falzon-Ojo, Oliver Nelso...",United Kingdom,2020-12-14,2021,TV-Y7,2 Seasons,Kids' TV,"Fearless, free-spirited Hilda finds new friend...",2020.0,12.0,December,-1.0,NaN,2.0
1551,s1552,TV Show,Hilda,unknown,"Bella Ramsey, Ameerah Falzon-Ojo, Oliver Nelso...",Canada,2020-12-14,2021,TV-Y7,2 Seasons,Kids' TV,"Fearless, free-spirited Hilda finds new friend...",2020.0,12.0,December,-1.0,NaN,2.0
1551,s1552,TV Show,Hilda,unknown,"Bella Ramsey, Ameerah Falzon-Ojo, Oliver Nelso...",United States,2020-12-14,2021,TV-Y7,2 Seasons,Kids' TV,"Fearless, free-spirited Hilda finds new friend...",2020.0,12.0,December,-1.0,NaN,2.0
1696,s1697,TV Show,Polly Pocket,unknown,"Emily Tennant, Shannon Chan-Kent, Kazumi Evans...",Canada,2020-11-15,2021,TV-Y,2 Seasons,Kids' TV,After uncovering a magical locket that allows ...,2020.0,11.0,November,-1.0,NaN,2.0
1696,s1697,TV Show,Polly Pocket,unknown,"Emily Tennant, Shannon Chan-Kent, Kazumi Evans...",United States,2020-11-15,2021,TV-Y,2 Seasons,Kids' TV,After uncovering a magical locket that allows ...,2020.0,11.0,November,-1.0,NaN,2.0
1696,s1697,TV Show,Polly Pocket,unknown,"Emily Tennant, Shannon Chan-Kent, Kazumi Evans...",Ireland,2020-11-15,2021,TV-Y,2 Seasons,Kids' TV,After uncovering a magical locket that allows ...,2020.0,11.0,November,-1.0,NaN,2.0
2920,s2921,TV Show,Love Is Blind,unknown,"Nick Lachey, Vanessa Lachey",United States,2020-02-13,2021,TV-MA,1 Season,Reality TV,Nick and Vanessa Lachey host this social exper...,2020.0,2.0,February,-1.0,NaN,1.0
2920,s2921,TV Show,Love Is Blind,unknown,"Nick Lachey, Vanessa Lachey",United States,2020-02-13,2021,TV-MA,1 Season,Romantic TV Shows,Nick and Vanessa Lachey host this social exper...,2020.0,2.0,February,-1.0,NaN,1.0
3168,s3169,TV Show,Fuller House,unknown,"Candace Cameron Bure, Jodie Sweetin, Andrea Ba...",United States,2019-12-06,2020,TV-PG,5 Seasons,TV Comedies,The Tanner familyâs adventures continue as D...,2019.0,12.0,December,-1.0,NaN,5.0
3287,s3288,TV Show,Maradona in Mexico,unknown,Diego Armando Maradona,Argentina,2019-11-13,2020,TV-MA,1 Season,Docuseries,"In this docuseries, soccer great Diego Maradon...",2019.0,11.0,November,-1.0,NaN,1.0


In [37]:
df.to_csv("netflix_cleaned2.csv", index=False)


In [35]:
df['duration_minutes'].describe()
df['seasons'].describe()


count    6881.000000
mean        1.757739
std         1.594556
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        17.000000
Name: seasons, dtype: float64